# Atari - BreakOut

## Imports

In [ ]:
import os
import sys
import warnings
from functools import partial

In [ ]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from torchrl.envs import GymWrapper
from torchrl.modules import QValueActor, DuelingCnnDQNet, EGreedyModule
from torchrl.data.replay_buffers import LazyMemmapStorage, PrioritizedReplayBuffer
from torchrl.record import CSVLogger

In [ ]:
from typing import Callable, Optional

In [ ]:
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from stable_baselines3.common.atari_wrappers import FireResetEnv, EpisodicLifeEnv, ClipRewardEnv

In [ ]:
warnings.filterwarnings("ignore")
gym.register_envs(ale_py)
if os.path.abspath("package") not in sys.path: sys.path.append(os.path.abspath("package"))

In [ ]:
from package.environment import GymPreprocessing, create_breakout_env
from package.dqn_types import ModelParameters, PathsParameters, EnvSpaceName
from package.utils import fill_buffer, init_collector
from package.video import VideoPlayer, Recorder, unstack_frames
from package.Logger import SmartLogger, LogsConfig
from package.modules import (Model,
                             Scale,
                             Optimizer,
                             Trainer,
                             initialize_weights,
                             init_lazy_layers,
                             n_parameters)

**In this version we use torchrl API to train a DQN agent on the Atari game `Breakout`.**

## Setup Environment.

**For wrapping `gym.Env` we use torchrl `Gym Wrapper`.**<br>
**The returned value of the environment variable is `uint8`, this will save a lot of memory for further storage.**<br>
**Note: all data is stored on `CPU`, and transfered to `MPS` or `GPU` only when needed.
The reason for this is that I had some issues with `MPS` and `GPU` memory management then working with torchrl API.**

## Hyperparameters

**To avoid problems with namespace overflow we will use special variables with hyperparameters.**

In [ ]:
model_space: ModelParameters = ModelParameters(
    n_frames=4,
    n_epochs=2 * 10 ** 5,
    batch_size=64,
    # rb_expansion=64 ensures that we collect as many new frames as we sample in one batch,
    # keeping the data distribution fresh.
    rb_expansion=64,
    lr=1e-4,
    min_lr=1e-5,
    weight_decay=1e-5,
    # We set max_grad_norm=1.0 to prevent gradient explosions and stabilize training.
    max_grad_norm=1.,
    soft_update_eps=0.995
)
paths_space: PathsParameters = PathsParameters(exp_name="dqn", log_dir="breakout_logs")
names_space: EnvSpaceName = EnvSpaceName()

In [ ]:
print(model_space)
print(paths_space)
print(names_space)

**For logging results will and custom-made `Logger`.**

Visually, one can observe that the "trail" of frames makes it difficult to assess exactly where the ball is heading. <br>
For the algorithm to see the direction of the ball, the residual movement should be less saturated (pixels should be darker). <br>
In practice, things are not straightforward. The ball can occupy several pixels at any given moment. <br>
Sometimes the pixels are indeed darker, and sometimes all pixels of the ball are equal or take close values chaotically. <br>
As a result, the model may frequently change its "opinion" about the ball's movement, failing to find an optimal game strategy.

This assessment was formed after observing the results of a trained DQN (CNN) model. <br>
The model would drive the controlled "paddle" into the corner of the screen, <br>
occasionally twitching in different directions but immediately returning to the corner. <br>
After visual analysis, I hypothesized that the model was getting stuck in the corner due to incorrect motion direction detection. <br>
Occasionally, "enlightenment" would occur, and the model would move towards the ball, <br>
but then lose its direction again on the next prediction and return to the corner.

To visually observe the ball's movement, I recommend using "plotly" <br>
due to the interactive capabilities of the created objects. <br>
One of these features allows checking pixel brightness without writing additional code.

In [ ]:
def explore_statistics(tensordict: TensorDict, key: str) -> dict[str, float]:
    """
    Calculates basic statistics (mean, std, min, max) for a specific tensor in a TensorDict.

    Args:
        tensordict: The TensorDict to analyze.
        key: The key of the tensor is to calculate statistics for.

    Returns:
        A dictionary containing the calculated statistics.
    """
    mean: float = tensordict[key].float().mean().item()
    std: float = tensordict[key].float().std().item()
    min_value: float = tensordict[key].float().min().item()
    max_value: float = tensordict[key].float().max().item()
    return dict(mean=mean, std=std, min=min_value, max=max_value)

We will use Reward Clipping during training.
Why?

1. Gradient Stability:
In Breakout, rewards for breaking bricks can increase as the game progresses
(e.g., lower rows — 1 point, upper rows — 7 points).
If the agent suddenly receives 7 points instead of 1, the `TD-error` spikes significantly.
This leads to massive weight updates that can destabilize the training
and potentially ruin an already learned strategy.

2. Value Function (V or Q) Scaling:
Algorithms estimate the expected sum of future rewards. If rewards aren't scaled,
`Q(s, a)` values can reach hundreds. Optimizers like Adam perform better
when target values are in a small, consistent range (e.g., -1 to 1 or around 0).

Why Clipping over Scaling?
It provides maximum stability. The agent simply learns: "hitting a brick is good,"
regardless of the specific point value.

Is a Penalty Loss Necessary? <br>
Why would it help? <br>
The agent lives in a world of "free risk." If it misses the ball, it simply stops receiving points. <br>
For the agent, this is a "zero." But at the beginning of training, it almost always gets "zeros" anyway <br>
because it hasn't learned how to hit the ball yet. <br>
Negative rewards create a clear contrast: <br>
• Hit the ball/break a brick: +1 (Good). <br>
• Miss the ball: -1 (Very Bad). <br>

This forces the agent to learn the fundamental rule much faster: "keep the paddle under the ball," <br>
before it even starts consciously aiming for the bricks. <br>

However, there are implementation challenges. If the penalty is too high, the agent might become "pessimistic." <br>
In some games, this leads to "suicide" strategies where the agent prefers to end the episode as quickly as possible <br>
to stop receiving small time-based penalties. <br>

To avoid experimenting with different penalty weights, we use a simpler trick: "End the episode on life loss." <br>
Result: For the agent, "death" (losing a life) is equivalent to losing all future potential rewards. <br>
This is the strongest signal for convergence, and it doesn't require tuning a penalty weight.

In [ ]:
# In our preprocessing, we use frame_skip=4 (Atari standard).
# Higher values like 5-7 make the ball movement too choppy for the model to track.
env_prep = GymPreprocessing(
    partial(AtariPreprocessing, noop_max=20, frame_skip=4, terminal_on_life_loss=False, screen_size=84),
    # EpisodicLifeEnv provides the "end episode on life loss" signal for faster training,
    # while AtariPreprocessing(terminal_on_life_loss=False) ensures reset() behaves correctly.
    partial(EpisodicLifeEnv),
    partial(FireResetEnv),
    partial(ClipRewardEnv),
    partial(FrameStackObservation, stack_size=model_space.n_frames)
)

In [ ]:
build_env: Callable[[], GymWrapper] = lambda: create_breakout_env(transform=env_prep)
envir: GymWrapper = build_env()

In [ ]:
logs_config: LogsConfig = LogsConfig(
    log_dir=paths_space.log_dir,
    metrics_save_freq=100,
    weights_save_freq=1000,
    videos_save_freq=1000
)
logger: SmartLogger = SmartLogger(names_space.actor, options=logs_config, exp_name=paths_space.exp_name)

## Evaluation & Visual Analysis
**Before training, we can visualize the agent's initial performance and analyze the observation data.**

In [ ]:
exp: TensorDict = envir.rollout(max_steps=400, auto_reset=True, break_when_any_done=False)
print(f"Observation shape: {exp[names_space.observation].shape}")
print(f"Observation statistics: {explore_statistics(exp, names_space.observation)}")

In [ ]:
# Experience TensorDict have duplicated observation state.
# For env0 we choose some action and got env1,
# after that we choose some action for state env1 and got state env2,
# and so on...
assert torch.all(exp[names_space.next][names_space.observation][0] == exp[names_space.observation][1])

In [ ]:
demo: bool = False
if demo:
    player = VideoPlayer(unstack_frames(exp, flip=True), fps=10, title="Atari Agent Preview")
    player.show(width=400, height=500)
del exp

**Data is not normalized before being fed to the model because we intend to store the "Buffer" as a "MemMap" object. <br>To reduce the buffer size, it is preferable to store images in "uint8" format.**

## Dueling DQN Configuration
**TorchRL provides implementations of Dueling CNN DQN. In this project, we utilize `EGreedyModule` for action exploration during environment interaction.**

In [ ]:
# We use a Dueling CNN architecture which is effective for Atari games.
# Note on Exploration: We originally considered NoisyLinear layers, but decided to use
# a classic EGreedyModule for more explicit control over the epsilon decay process.
cnn_kwargs = dict(num_cells=(32, 64, 128), kernel_sizes=(8, 4, 3), strides=(4, 2, 1))
mlp_kwargs = dict(num_cells=512)

In [ ]:
model: TensorDictModule = Model(
    scale=TensorDictModule(
        Scale(value=255.),
        in_keys=names_space.observation,
        out_keys=names_space.transformed
    ),
    actor=QValueActor(
        DuelingCnnDQNet(
            out_features=envir.action_spec.shape.numel(),
            cnn_kwargs=cnn_kwargs,
            mlp_kwargs=mlp_kwargs
        ),
        in_keys=[names_space.transformed],
        spec=envir.action_spec
    ),
    explorer=EGreedyModule(
        spec=envir.action_spec,
        # Starting with high exploration to discover initial rewards (breaking the first brick).
        eps_init=1.,
        eps_end=0.01,
        annealing_num_steps=100000  # Gradual decay over half the training epochs.
    )
)

## Weights Initialization
**Proper weight initialization is essential for deep reinforcement learning stability. We define a custom initialization function for various layer types.**

In [ ]:
model: TensorDictModule = init_lazy_layers(envir.reset(), model).apply(initialize_weights)
last_upd: Optional[str] = logger.get_last_update(names_space.actor)
model.load_state_dict(torch.load(last_upd)) if last_upd else None

In [ ]:
print(f"Weights counts: {n_parameters(model)}")

## Prioritized Replay Buffer & Collector.

**The classic Replay Buffer is not as efficient for training because sampling does not take into account the contribution of each individual record.** <br>
**Prioritized Replay Buffer compensates for this deficiency.
By adding new records to the buffer, we will also update the priority values of the remaining records based on the error received.
The higher the rating, the higher the priority. This way, sampling of records with a large error will occur more often.
The sampling algorithm is based on SumTree.**

**Records will be buffered and written to a dedicated directory.** <br>
**The size of the dedicated directory after the buffer is completely full will be over 200 GB.** <br>
**It is for this reason that the environment data is supplied in `uint8`.**

**Buffer will be automatically cleaned up after ending the session.**

In [ ]:
buffer = PrioritizedReplayBuffer(
    alpha=0.6,
    beta=0.5,
    batch_size=model_space.batch_size,
    storage=LazyMemmapStorage(
        max_size=5 * 10 ** 5,
        scratch_dir=paths_space.storage_path,
        existsok=True,
        auto_cleanup=True
    )
)

## Memory & Experience Collection
**Efficient experience storage is crucial for RL. We use a Prioritized Replay Buffer with memory-mapped storage to handle large datasets.**

In [ ]:
collector_kwargs = dict(
    frames_per_batch=model_space.batch_size,
    total_frames=-1,
    extend_buffer=False,
    storing_device=model_space.cpu,
    policy_device=model_space.dev
)

In [ ]:
_ = fill_buffer(
    init_collector(build_env, **collector_kwargs),
    buffer,
    # We pre-fill the buffer with 10^5 frames to provide a diverse initial dataset for the optimizer.
    1000,  # 10 ** 5,
    show=True
)

## Agent Configuration & Initialization
**Configure the collector and recording utilities, then initialize the optimizer and trainer.**

In [ ]:
collector_kwargs = dict(
    frames_per_batch=model_space.batch_size,
    total_frames=-1,
    extend_buffer=False,
    storing_device=model_space.cpu,
    policy_device=model_space.dev
)

In [ ]:
video_maker = Recorder(
    CSVLogger(paths_space.exp_name,
              paths_space.log_dir,
              video_format="mp4",
              video_fps=30),
    build_env(),
    deterministic=True
)
optim_method = Optimizer(network=model.to(model_space.dev), action_space=envir.action_spec, params=model_space)

**"video_maker" will shoot video of agent game session and save in log directory.**

## Training Loop & Optimization

**Torchrl has an API for model training. <br>
However, this API currently has several compatibility issues with other classes. <br>
Therefore, we're implementing our own functionality.**

**The custom Trainer class manages the training process, integrating the model, optimizer, and logging utilities.**

In [ ]:
trainer = Trainer(model, optim_method, model_space, names_space, logger, video_maker)

In [ ]:
trainer.train(
    # For a full training run, use model_space.n_epochs.
    n_epochs=10,  # model_space.n_epochs,
    rb=buffer,
    loader=init_collector(build_env, model, **collector_kwargs)
)

**Note: The Logger uses the term "Epoch" to refer to the training step index.**

In [ ]:
logger.draw_scalars()